# Etiquetado semántico — LIM

Complementa el Dataset A producido por `02_etiquetado_task1.ipynb`.
Usa búsqueda semántica (FAISS) + validación con LLM para encontrar fragmentos
de LIM que no tienen encabezado explícito y por eso quedan como UNKNOWN con regex.

Basado en el método de Mateo (`test_v5.ipynb`), adaptado para correr en Colab
con corpus en parquet y sin Ollama local.

**Requisito:** API key de Groq (gratis en console.groq.com)

## 1. Instalación

In [ ]:
!pip install -q faiss-cpu sentence-transformers groq pyarrow pandas tqdm

## 2. Montar Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Configuración

In [ ]:
# Groq API key — obtener gratis en console.groq.com
# Opcion A: pegar aqui directamente
GROQ_API_KEY = "gsk_REEMPLAZA_CON_TU_CLAVE"
# Opcion B (recomendada): usar Secrets de Colab
# from google.colab import userdata
# GROQ_API_KEY = userdata.get('GROQ_API_KEY')

GROQ_MODEL = "llama3-70b-8192"   # gratis, mejor calidad para español

INPUT_PATH       = "/content/drive/MyDrive/camilo_corpus.parquet"
DATASET_A_PATH   = "/content/drive/MyDrive/camilo_dataset_A.parquet"
OUTPUT_LIM_PATH  = "/content/drive/MyDrive/camilo_lim_llm.parquet"

META_LIM         = 2000   # fragmentos de entrenamiento que necesitamos
YA_TENGO_LIM     = 430    # fragmentos de LIM que ya salieron del notebook regex
FALTAN_LIM       = META_LIM - YA_TENGO_LIM

MIN_PALABRAS     = 250
MAX_PALABRAS     = 1000
CHUNK_SIZE       = 800    # palabras por chunk
CHUNK_OVERLAP    = 200
TOP_K            = 10     # candidatos por query semántica
SCORE_MIN        = 6      # umbral LLM (1-10)
SEED             = 42

print(f"Modelo:       {GROQ_MODEL}")
print(f"Ya tengo LIM: {YA_TENGO_LIM}")
print(f"Faltan LIM:   {FALTAN_LIM}")
print(f"Corpus:       {INPUT_PATH}")

## 4. Imports

In [ ]:
import json
import os
import re
import time
import numpy as np
import pandas as pd
import faiss
from groq import Groq
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm

groq_client = Groq(api_key=GROQ_API_KEY)

print("Cargando modelo de embeddings...")
embedding_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
print("Listo.")

## 5. Queries semánticas para LIM

In [ ]:
# Frases representativas de secciones de limitaciones en papers en español.
# No son patrones de regex — se usan como vectores de búsqueda semántica.
QUERIES_LIM = [
    "una limitación de este estudio es",
    "entre las limitaciones de esta investigación",
    "este trabajo no considera",
    "queda fuera del alcance de este estudio",
    "como trabajo futuro se propone",
    "futuras investigaciones podrían explorar",
    "este enfoque tiene la restricción de",
    "no fue posible abordar en este trabajo",
    "líneas de investigación futuras incluyen",
    "requiere mayor investigación en el futuro",
    "el tamaño de la muestra limita",
    "los resultados no pueden generalizarse",
]

## 6. Funciones

In [ ]:
def split_into_chunks(text, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    words = text.split()
    chunks, start = [], 0
    step = chunk_size - overlap
    while start < len(words):
        chunks.append(" ".join(words[start:start + chunk_size]))
        start += step
    return chunks


def word_count(text):
    return len(text.split())


def expand_chunk(chunks, idx, min_w=MIN_PALABRAS, max_w=MAX_PALABRAS):
    current = chunks[idx]
    total = word_count(current)
    left, right = idx - 1, idx + 1
    while total < min_w and (left >= 0 or right < len(chunks)):
        if left >= 0:
            c = chunks[left]
            if total + word_count(c) <= max_w:
                current = c + " " + current
                total += word_count(c)
                left -= 1
            else:
                left = -1
        if total >= min_w:
            break
        if right < len(chunks):
            c = chunks[right]
            if total + word_count(c) <= max_w:
                current = current + " " + c
                total += word_count(c)
                right += 1
            else:
                right = len(chunks)
        if left < 0 and right >= len(chunks):
            break
    return current


def build_faiss_index(chunks):
    emb = embedding_model.encode(chunks, normalize_embeddings=True, show_progress_bar=False)
    index = faiss.IndexFlatIP(emb.shape[1])
    index.add(np.array(emb, dtype="float32"))
    return index, emb


def semantic_search(index, query, top_k=TOP_K):
    q = embedding_model.encode([query], normalize_embeddings=True)
    distances, indices = index.search(np.array(q, dtype="float32"), top_k)
    return list(zip(indices[0], distances[0]))


def classify_with_llm(text, retries=3):
    prompt = f"""Eres un experto en análisis del discurso científico en español.

Determina si el siguiente fragmento de un artículo científico pertenece a una sección
de LIMITACIONES o TRABAJO FUTURO. Evalúa la función retórica del texto, no si contiene
palabras clave específicas.

Una sección de limitaciones: reconoce restricciones del estudio, señala qué no se pudo
abordar, indica que los resultados no son generalizables, o propone trabajo futuro.

Fragmento:
\"\"\"{text[:2500]}\"\"\"

Responde ÚNICAMENTE con JSON válido:
{{"es_lim": true | false, "score": número entero 1-10, "razon": "una oración"}}"""

    for attempt in range(retries):
        try:
            resp = groq_client.chat.completions.create(
                model=GROQ_MODEL,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.1,
                max_tokens=120,
            )
            raw = resp.choices[0].message.content.strip()
            match = re.search(r"\{.*\}", raw, re.DOTALL)
            if match:
                return json.loads(match.group())
        except Exception as e:
            time.sleep(2 ** attempt)
    return None

## 7. Cargar corpus y excluir docs ya etiquetados

In [ ]:
df_corpus = pd.read_parquet(INPUT_PATH)
df_corpus = df_corpus.sample(frac=1, random_state=SEED).reset_index(drop=True)

# Excluir documentos que ya aportaron fragmentos LIM en el Dataset A
if os.path.exists(DATASET_A_PATH):
    df_a = pd.read_parquet(DATASET_A_PATH)
    docs_ya_lim = set(df_a[df_a["etiqueta_auto"] == "LIM"]["doc_id"].tolist())
    df_corpus = df_corpus[~df_corpus["doc_id"].isin(docs_ya_lim)].reset_index(drop=True)
    print(f"Documentos excluidos (ya tienen LIM en Dataset A): {len(docs_ya_lim):,}")

print(f"Documentos disponibles para buscar: {len(df_corpus):,}")

## 8. Pipeline — búsqueda semántica + validación LLM

In [ ]:
fragmentos_lim = []
textos_vistos  = set()

pbar = tqdm(df_corpus.itertuples(index=False), total=len(df_corpus), desc="Buscando LIM")

for row in pbar:
    if len(fragmentos_lim) >= FALTAN_LIM:
        print(f"\nCuota alcanzada: {len(fragmentos_lim)} fragmentos de LIM.")
        break

    texto = row.texto.strip()
    if len(texto.split()) < MIN_PALABRAS:
        continue

    chunks = split_into_chunks(texto)
    if len(chunks) < 2:
        continue

    index, _ = build_faiss_index(chunks)

    # Recuperar candidatos para cada query
    candidatos = {}
    for query in QUERIES_LIM:
        for idx, score in semantic_search(index, query):
            if idx not in candidatos or score > candidatos[idx]:
                candidatos[idx] = float(score)

    # Ordenar por score semántico descendente
    for idx, sem_score in sorted(candidatos.items(), key=lambda x: -x[1]):
        if len(fragmentos_lim) >= FALTAN_LIM:
            break

        fragmento = expand_chunk(chunks, idx)
        wc = word_count(fragmento)
        if wc < MIN_PALABRAS or wc > MAX_PALABRAS:
            continue
        if fragmento in textos_vistos:
            continue
        textos_vistos.add(fragmento)

        resultado = classify_with_llm(fragmento)
        if resultado is None:
            continue

        if resultado.get("es_lim") and resultado.get("score", 0) >= SCORE_MIN:
            fragmentos_lim.append({
                "doc_id":        str(row.doc_id),
                "fragmento_id":  f"{row.doc_id}_llm_{len(fragmentos_lim):04d}",
                "texto":         fragmento,
                "etiqueta_auto": "LIM",
                "llm_score":     resultado["score"],
                "llm_razon":     resultado.get("razon", ""),
                "sem_score":     sem_score,
                "num_palabras":  wc,
                "asignado_a":    "Camilo",
            })

        time.sleep(0.5)  # respetar rate limit de Groq

    pbar.set_postfix_str(f"LIM encontrados: {len(fragmentos_lim)}/{FALTAN_LIM}")

print(f"\nTotal fragmentos LIM encontrados: {len(fragmentos_lim):,}")

## 9. Guardar

In [ ]:
df_lim = pd.DataFrame(fragmentos_lim)
df_lim.to_parquet(OUTPUT_LIM_PATH, index=False)

mb = os.path.getsize(OUTPUT_LIM_PATH) / (1024 ** 2)
print(f"Guardado en: {OUTPUT_LIM_PATH}")
print(f"Fragmentos: {len(df_lim):,}")
print(f"Tamaño:     {mb:.1f} MB")
print()
print("Score LLM promedio:", round(df_lim['llm_score'].mean(), 2))
df_lim[["doc_id", "llm_score", "num_palabras", "llm_razon"]].head(5)

## 10. Combinar con Dataset A

In [ ]:
# Une los fragmentos LIM nuevos con el Dataset A existente
# para producir un Dataset A completo con LIM >= 2000

OUTPUT_A_COMPLETO = "/content/drive/MyDrive/camilo_dataset_A_completo.parquet"

df_a = pd.read_parquet(DATASET_A_PATH)
df_a_completo = pd.concat([df_a, df_lim], ignore_index=True)
df_a_completo = df_a_completo.drop_duplicates(subset=["fragmento_id"]).reset_index(drop=True)

df_a_completo.to_parquet(OUTPUT_A_COMPLETO, index=False)

print("Dataset A completo:")
print(df_a_completo["etiqueta_auto"].value_counts().to_string())
print()
print(f"Guardado en: {OUTPUT_A_COMPLETO}")